# PART 2 — 3D Patch Generation + Data Loader + 3D U-Net Training

## Project
**3D Kidney and Kidney Tumor Segmentation Using Deep Learning: A Comparative Study on the KiTS23 Dataset**

This notebook performs:

- KiTS23 case discovery
- CT and segmentation loading
- 64×64×64 3D patch extraction
- Background-aware patch selection
- Train/validation case split
- PyTorch Dataset and DataLoader creation
- 3D U-Net implementation
- Combined Dice + Cross-Entropy loss
- Dice metric calculation
- 10-epoch training
- Validation after every epoch
- Best model checkpoint saving
- Training history export
- Training Loss and Dice graphs

In [ ]:

# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import os
import glob
import csv
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


In [ ]:

# ==========================================
# 2. PROJECT PATHS AND CONFIGURATION
# ==========================================

PROJECT_PATH = r"C:\Users\USER\Kidney_Segmentation_Project"

DATA_PATH = os.path.join(PROJECT_PATH, "data")
PROCESSED_PATH = os.path.join(PROJECT_PATH, "processed_data")
MODEL_PATH = os.path.join(PROJECT_PATH, "models")
LOG_PATH = os.path.join(PROJECT_PATH, "logs")
RESULT_PATH = os.path.join(PROJECT_PATH, "results")
FIGURE_PATH = os.path.join(PROJECT_PATH, "figures")

for p in [PROCESSED_PATH, MODEL_PATH, LOG_PATH, RESULT_PATH, FIGURE_PATH]:
    os.makedirs(p, exist_ok=True)

PATCH_SIZE = (64, 64, 64)
NUM_CLASSES = 4
EPOCHS = 10
BATCH_SIZE = 1
LEARNING_RATE = 1e-4
NUM_WORKERS = 0  # 
VAL_FRACTION = 0.20
SEED = 42

# Set True 
KEEP_FOREGROUND_ONLY = True

# Set to an integer such as 20 for debugging.
# 
MAX_CASES = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project:", PROJECT_PATH)
print("Patch size:", PATCH_SIZE)
print("Epochs:", EPOCHS)


In [ ]:

# ==========================================
# 3. DISCOVER LABELLED KiTS23 CASES
# ==========================================

all_case_dirs = sorted(
    glob.glob(os.path.join(DATA_PATH, "case_*"))
)

labelled_cases = []

for case_dir in all_case_dirs:
    image_path = os.path.join(case_dir, "imaging.nii.gz")
    mask_path = os.path.join(case_dir, "segmentation.nii.gz")

    if os.path.exists(image_path) and os.path.exists(mask_path):
        labelled_cases.append(case_dir)

if MAX_CASES is not None:
    labelled_cases = labelled_cases[:MAX_CASES]

print("Total case folders found:", len(all_case_dirs))
print("Labelled cases available for supervised training:", len(labelled_cases))

if len(labelled_cases) == 0:
    raise RuntimeError(
        "No labelled KiTS23 cases were found. "
        "Check DATA_PATH and your case folder structure."
    )


In [ ]:

# ==========================================
# 4. CASE-LEVEL TRAIN / VALIDATION SPLIT
# ==========================================

rng = np.random.default_rng(SEED)
indices = np.arange(len(labelled_cases))
rng.shuffle(indices)

split_index = int(len(indices) * (1 - VAL_FRACTION))

train_indices = indices[:split_index]
val_indices = indices[split_index:]

train_cases = [labelled_cases[i] for i in train_indices]
val_cases = [labelled_cases[i] for i in val_indices]

print("Training cases:", len(train_cases))
print("Validation cases:", len(val_cases))

print("\nFirst training case:")
print(train_cases[0])

print("\nFirst validation case:")
print(val_cases[0] if len(val_cases) > 0 else "No validation case")


In [ ]:

# ==========================================
# 5. NIFTI LOADING + CT NORMALIZATION
# ==========================================

def load_nifti(path):
    nii = nib.load(path)
    data = nii.get_fdata()
    return data


def normalize_ct(volume, lower_hu=-200, upper_hu=300):
    """
    Clip CT intensities to a kidney-relevant HU window
    and scale into [0, 1].
    """
    volume = np.asarray(volume, dtype=np.float32)

    volume = np.clip(
        volume,
        lower_hu,
        upper_hu
    )

    volume = (
        volume - lower_hu
    ) / (
        upper_hu - lower_hu
    )

    return volume.astype(np.float32)


sample_case = train_cases[0]

sample_ct = load_nifti(
    os.path.join(sample_case, "imaging.nii.gz")
)

sample_mask = load_nifti(
    os.path.join(sample_case, "segmentation.nii.gz")
).astype(np.int16)

print("Sample CT shape:", sample_ct.shape)
print("Sample mask shape:", sample_mask.shape)
print("Labels:", np.unique(sample_mask))


In [ ]:

# ==========================================
# 6. 3D PATCH START COORDINATES
# ==========================================

def axis_starts(length, patch_length):
    """
    Generate non-overlapping starts and always include
    the final patch so the end of the volume is covered.
    """
    if length <= patch_length:
        return [0]

    starts = list(range(0, length - patch_length + 1, patch_length))

    last_start = length - patch_length

    if starts[-1] != last_start:
        starts.append(last_start)

    return starts


def pad_to_patch_size(volume, mask, patch_size=(64, 64, 64)):
    """
    Pad small volumes so each dimension is at least PATCH_SIZE.
    """
    padding = []

    for current, required in zip(volume.shape, patch_size):
        missing = max(required - current, 0)
        before = missing // 2
        after = missing - before
        padding.append((before, after))

    if any(p != (0, 0) for p in padding):
        volume = np.pad(
            volume,
            padding,
            mode="constant",
            constant_values=0
        )

        mask = np.pad(
            mask,
            padding,
            mode="constant",
            constant_values=0
        )

    return volume, mask


In [ ]:

# ==========================================
# 7. BUILD PATCH INDEX
# ==========================================

def build_patch_index(
    case_dirs,
    patch_size=(64, 64, 64),
    foreground_only=True
):
    

    patch_records = []

    pd, ph, pw = patch_size

    for case_dir in tqdm(case_dirs, desc="Indexing cases"):

        mask_path = os.path.join(case_dir, "segmentation.nii.gz")

        mask = load_nifti(mask_path).astype(np.int16)

        # determine padded shape only
        dummy = np.zeros(mask.shape, dtype=np.uint8)
        dummy, mask = pad_to_patch_size(
            dummy,
            mask,
            patch_size
        )

        z_starts = axis_starts(mask.shape[0], pd)
        y_starts = axis_starts(mask.shape[1], ph)
        x_starts = axis_starts(mask.shape[2], pw)

        for z in z_starts:
            for y in y_starts:
                for x in x_starts:

                    mask_patch = mask[
                        z:z+pd,
                        y:y+ph,
                        x:x+pw
                    ]

                    if foreground_only and not np.any(mask_patch > 0):
                        continue

                    patch_records.append(
                        {
                            "case_dir": case_dir,
                            "z": int(z),
                            "y": int(y),
                            "x": int(x)
                        }
                    )

    return patch_records


train_patch_records = build_patch_index(
    train_cases,
    PATCH_SIZE,
    foreground_only=KEEP_FOREGROUND_ONLY
)

val_patch_records = build_patch_index(
    val_cases,
    PATCH_SIZE,
    foreground_only=KEEP_FOREGROUND_ONLY
)

print("Training patches:", len(train_patch_records))
print("Validation patches:", len(val_patch_records))


In [ ]:

# ==========================================
# 8. PYTORCH DATASET
# ==========================================

class KiTS23PatchDataset(Dataset):

    def __init__(
        self,
        patch_records,
        patch_size=(64, 64, 64)
    ):

        self.patch_records = patch_records
        self.patch_size = patch_size

        # Simple one-case cache
        self._cached_case = None
        self._cached_ct = None
        self._cached_mask = None


    def __len__(self):
        return len(self.patch_records)


    def _load_case(self, case_dir):

        if self._cached_case == case_dir:
            return self._cached_ct, self._cached_mask

        ct_path = os.path.join(
            case_dir,
            "imaging.nii.gz"
        )

        mask_path = os.path.join(
            case_dir,
            "segmentation.nii.gz"
        )

        ct = load_nifti(ct_path)
        mask = load_nifti(mask_path).astype(np.int16)

        ct = normalize_ct(ct)

        ct, mask = pad_to_patch_size(
            ct,
            mask,
            self.patch_size
        )

        self._cached_case = case_dir
        self._cached_ct = ct
        self._cached_mask = mask

        return ct, mask


    def __getitem__(self, index):

        record = self.patch_records[index]

        ct, mask = self._load_case(
            record["case_dir"]
        )

        z = record["z"]
        y = record["y"]
        x = record["x"]

        pd, ph, pw = self.patch_size

        image_patch = ct[
            z:z+pd,
            y:y+ph,
            x:x+pw
        ]

        mask_patch = mask[
            z:z+pd,
            y:y+ph,
            x:x+pw
        ]

        # [D,H,W] -> [C,D,H,W]
        image_patch = np.expand_dims(
            image_patch,
            axis=0
        )

        image_tensor = torch.from_numpy(
            image_patch.astype(np.float32)
        )

        mask_tensor = torch.from_numpy(
            mask_patch.astype(np.int64)
        )

        return image_tensor, mask_tensor


train_dataset = KiTS23PatchDataset(
    train_patch_records,
    PATCH_SIZE
)

val_dataset = KiTS23PatchDataset(
    val_patch_records,
    PATCH_SIZE
)

print("Training dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))


In [ ]:

# ==========================================
# 9. DATALOADERS
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

first_images, first_masks = next(iter(train_loader))

print("Image batch shape:", first_images.shape)
print("Mask batch shape:", first_masks.shape)
print("Mask labels:", torch.unique(first_masks))


In [ ]:

# ==========================================
# 10. 3D U-NET ARCHITECTURE
# ==========================================

class DoubleConv3D(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.InstanceNorm3d(out_channels),

            nn.ReLU(inplace=True),

            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.InstanceNorm3d(out_channels),

            nn.ReLU(inplace=True)
        )


    def forward(self, x):
        return self.block(x)


class UNet3D(nn.Module):

    def __init__(
        self,
        in_channels=1,
        num_classes=4,
        base_filters=16
    ):

        super().__init__()

        f = base_filters

        self.enc1 = DoubleConv3D(
            in_channels,
            f
        )

        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = DoubleConv3D(
            f,
            f*2
        )

        self.pool2 = nn.MaxPool3d(2)

        self.enc3 = DoubleConv3D(
            f*2,
            f*4
        )

        self.pool3 = nn.MaxPool3d(2)

        self.bottleneck = DoubleConv3D(
            f*4,
            f*8
        )

        self.up3 = nn.ConvTranspose3d(
            f*8,
            f*4,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv3D(
            f*8,
            f*4
        )

        self.up2 = nn.ConvTranspose3d(
            f*4,
            f*2,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv3D(
            f*4,
            f*2
        )

        self.up1 = nn.ConvTranspose3d(
            f*2,
            f,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv3D(
            f*2,
            f
        )

        self.output = nn.Conv3d(
            f,
            num_classes,
            kernel_size=1
        )


    def forward(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool1(e1)
        )

        e3 = self.enc3(
            self.pool2(e2)
        )

        b = self.bottleneck(
            self.pool3(e3)
        )

        d3 = self.up3(b)

        d3 = torch.cat(
            [d3, e3],
            dim=1
        )

        d3 = self.dec3(d3)

        d2 = self.up2(d3)

        d2 = torch.cat(
            [d2, e2],
            dim=1
        )

        d2 = self.dec2(d2)

        d1 = self.up1(d2)

        d1 = torch.cat(
            [d1, e1],
            dim=1
        )

        d1 = self.dec1(d1)

        return self.output(d1)


model = UNet3D(
    in_channels=1,
    num_classes=NUM_CLASSES,
    base_filters=16
).to(DEVICE)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(model)
print("\nTotal parameters:", total_params)
print("Trainable parameters:", trainable_params)


In [ ]:

# ==========================================
# 11. DICE LOSS + CROSS ENTROPY LOSS
# ==========================================

cross_entropy = nn.CrossEntropyLoss()


def multiclass_dice_loss(
    logits,
    targets,
    num_classes=4,
    smooth=1e-5,
    include_background=False
):
    """
    Multi-class soft Dice loss.
    """

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    one_hot = torch.nn.functional.one_hot(
        targets,
        num_classes=num_classes
    )

    # [B,D,H,W,C] -> [B,C,D,H,W]
    one_hot = one_hot.permute(
        0, 4, 1, 2, 3
    ).float()

    if include_background:
        probs_used = probabilities
        target_used = one_hot
    else:
        probs_used = probabilities[:, 1:]
        target_used = one_hot[:, 1:]

    dims = (0, 2, 3, 4)

    intersection = torch.sum(
        probs_used * target_used,
        dim=dims
    )

    denominator = torch.sum(
        probs_used,
        dim=dims
    ) + torch.sum(
        target_used,
        dim=dims
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return 1.0 - dice.mean()


def combined_loss(logits, targets):

    ce = cross_entropy(
        logits,
        targets
    )

    dice = multiclass_dice_loss(
        logits,
        targets,
        num_classes=NUM_CLASSES,
        include_background=False
    )

    return ce + dice


In [ ]:

# ==========================================
# 12. DICE METRIC
# ==========================================

@torch.no_grad()
def dice_scores_from_logits(
    logits,
    targets,
    num_classes=4,
    smooth=1e-5
):
    """
    Returns:
    overall foreground Dice,
    kidney Dice,
    tumour Dice,
    cyst Dice.
    """

    predictions = torch.argmax(
        logits,
        dim=1
    )

    per_class = {}

    for class_id, class_name in [
        (1, "kidney"),
        (2, "tumour"),
        (3, "cyst")
    ]:

        pred_class = (
            predictions == class_id
        ).float()

        target_class = (
            targets == class_id
        ).float()

        intersection = torch.sum(
            pred_class * target_class
        )

        denominator = (
            torch.sum(pred_class)
            +
            torch.sum(target_class)
        )

        dice = (
            2.0 * intersection + smooth
        ) / (
            denominator + smooth
        )

        per_class[class_name] = dice.item()

    overall = np.mean(
        [
            per_class["kidney"],
            per_class["tumour"],
            per_class["cyst"]
        ]
    )

    per_class["overall"] = float(overall)

    return per_class


In [ ]:

# ==========================================
# 13. OPTIMIZER
# ==========================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)


In [ ]:

# ==========================================
# 14. TRAINING + VALIDATION FUNCTIONS
# ==========================================

def run_train_epoch(
    model,
    loader,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    running_dice = 0.0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for images, masks in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(images)

        loss = combined_loss(
            logits,
            masks
        )

        loss.backward()

        optimizer.step()

        scores = dice_scores_from_logits(
            logits.detach(),
            masks,
            NUM_CLASSES
        )

        running_loss += loss.item()

        running_dice += scores["overall"]

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            dice=f"{scores['overall']:.4f}"
        )

    return (
        running_loss / len(loader),
        running_dice / len(loader)
    )


@torch.no_grad()
def run_validation_epoch(
    model,
    loader,
    device
):

    model.eval()

    running_loss = 0.0
    running_dice = 0.0

    kidney_scores = []
    tumour_scores = []
    cyst_scores = []

    progress = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    for images, masks in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        logits = model(images)

        loss = combined_loss(
            logits,
            masks
        )

        scores = dice_scores_from_logits(
            logits,
            masks,
            NUM_CLASSES
        )

        running_loss += loss.item()

        running_dice += scores["overall"]

        kidney_scores.append(
            scores["kidney"]
        )

        tumour_scores.append(
            scores["tumour"]
        )

        cyst_scores.append(
            scores["cyst"]
        )

    metrics = {
        "loss":
            running_loss / len(loader),

        "overall_dice":
            running_dice / len(loader),

        "kidney_dice":
            float(np.mean(kidney_scores)),

        "tumour_dice":
            float(np.mean(tumour_scores)),

        "cyst_dice":
            float(np.mean(cyst_scores))
    }

    return metrics


In [ ]:

# ==========================================
# 15. TRAIN 3D U-NET FOR 10 EPOCHS
# ==========================================

history = {
    "epoch": [],
    "train_loss": [],
    "train_dice": [],
    "val_loss": [],
    "val_dice": [],
    "val_kidney_dice": [],
    "val_tumour_dice": [],
    "val_cyst_dice": []
}

best_val_dice = -1.0

best_model_file = os.path.join(
    MODEL_PATH,
    "best_3D_UNet_model.pt"
)

last_model_file = os.path.join(
    MODEL_PATH,
    "last_3D_UNet_model.pt"
)

for epoch in range(1, EPOCHS + 1):

    print(
        f"\n========== Epoch {epoch}/{EPOCHS} =========="
    )

    train_loss, train_dice = run_train_epoch(
        model,
        train_loader,
        optimizer,
        DEVICE
    )

    val_metrics = run_validation_epoch(
        model,
        val_loader,
        DEVICE
    )

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_dice"].append(train_dice)
    history["val_loss"].append(
        val_metrics["loss"]
    )
    history["val_dice"].append(
        val_metrics["overall_dice"]
    )
    history["val_kidney_dice"].append(
        val_metrics["kidney_dice"]
    )
    history["val_tumour_dice"].append(
        val_metrics["tumour_dice"]
    )
    history["val_cyst_dice"].append(
        val_metrics["cyst_dice"]
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Dice: {train_dice:.4f}"
    )

    print(
        f"Val Loss: {val_metrics['loss']:.4f}"
    )

    print(
        f"Val Overall Dice: {val_metrics['overall_dice']:.4f}"
    )

    print(
        f"Val Kidney Dice: {val_metrics['kidney_dice']:.4f}"
    )

    print(
        f"Val Tumour Dice: {val_metrics['tumour_dice']:.4f}"
    )

    print(
        f"Val Cyst Dice: {val_metrics['cyst_dice']:.4f}"
    )

    if val_metrics["overall_dice"] > best_val_dice:

        best_val_dice = val_metrics["overall_dice"]

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "best_val_dice":
                    best_val_dice,
                "patch_size":
                    PATCH_SIZE,
                "num_classes":
                    NUM_CLASSES
            },
            best_model_file
        )

        print(
            "Best model updated:",
            best_model_file
        )


torch.save(
    {
        "epoch": EPOCHS,
        "model_state_dict":
            model.state_dict(),
        "optimizer_state_dict":
            optimizer.state_dict(),
        "patch_size":
            PATCH_SIZE,
        "num_classes":
            NUM_CLASSES
    },
    last_model_file
)

print("\nTraining completed.")
print("Best validation Dice:", best_val_dice)


In [ ]:

# ==========================================
# 16. SAVE TRAINING HISTORY AS CSV
# ==========================================

history_csv = os.path.join(
    LOG_PATH,
    "3D_UNet_10_epoch_history.csv"
)

with open(
    history_csv,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow(
        [
            "Epoch",
            "Training Loss",
            "Training Dice",
            "Validation Loss",
            "Validation Dice",
            "Kidney Dice",
            "Tumour Dice",
            "Cyst Dice"
        ]
    )

    for i in range(len(history["epoch"])):

        writer.writerow(
            [
                history["epoch"][i],
                history["train_loss"][i],
                history["train_dice"][i],
                history["val_loss"][i],
                history["val_dice"][i],
                history["val_kidney_dice"][i],
                history["val_tumour_dice"][i],
                history["val_cyst_dice"][i]
            ]
        )

print("History saved to:")
print(history_csv)


In [ ]:

# ==========================================
# 17. TRAINING LOSS GRAPH
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    history["epoch"],
    history["train_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    history["epoch"],
    history["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("3D U-Net Training and Validation Loss Over 10 Epochs")
plt.legend()
plt.grid(True)

loss_figure = os.path.join(
    FIGURE_PATH,
    "3D_UNet_Training_Validation_Loss.png"
)

plt.savefig(
    loss_figure,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved:", loss_figure)


In [ ]:

# ==========================================
# 18. DICE SCORE GRAPH
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    history["epoch"],
    history["train_dice"],
    marker="o",
    label="Training Dice"
)

plt.plot(
    history["epoch"],
    history["val_dice"],
    marker="o",
    label="Validation Dice"
)

plt.xlabel("Epoch")
plt.ylabel("Dice Similarity Coefficient")
plt.title("3D U-Net Dice Score Over 10 Epochs")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)

dice_figure = os.path.join(
    FIGURE_PATH,
    "3D_UNet_Training_Validation_Dice.png"
)

plt.savefig(
    dice_figure,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved:", dice_figure)


In [ ]:

# ==========================================
# 19. FINAL TRAINING SUMMARY
# ==========================================

print("==========================================")
print("3D U-Net 10-Epoch Training Summary")
print("==========================================")

print("Training cases:", len(train_cases))
print("Validation cases:", len(val_cases))

print("Training patches:", len(train_patch_records))
print("Validation patches:", len(val_patch_records))

print("Final training loss:", history["train_loss"][-1])
print("Final training Dice:", history["train_dice"][-1])

print("Final validation loss:", history["val_loss"][-1])
print("Final validation Dice:", history["val_dice"][-1])

print("Final kidney Dice:", history["val_kidney_dice"][-1])
print("Final tumour Dice:", history["val_tumour_dice"][-1])
print("Final cyst Dice:", history["val_cyst_dice"][-1])

print("\nBest model:")
print(best_model_file)
